# oneM2M Weather Station Demo

This notebook demonstrates a complete oneM2M data collection stack:

```
Weather Station (simulated, port 8001)
    | HTTP GET /temperature  (every 30s)
    v
oneM2M AE (WeatherStationAE)
    | POST ContentInstance
    v
ACME CSE (port 8082)
    ^
MCP Server -> ADK Agent  (covered in next notebook)
```

Run each cell in order from top to bottom.

## Step 1: Install dependencies

In [1]:
!pip install fastapi uvicorn httpx requests acmecse --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 701.8/701.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.4/285.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.9/148.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 1.8 MB/s eta 0:00:00


## Step 2: Start the simulated weather station

A FastAPI server on port 8001 that returns a realistic temperature reading on each request.
The value changes by at most +/- 0.5 degrees C per request, staying between 10 and 30 degrees C.

In [2]:
import random
import threading
import time
from fastapi import FastAPI
import uvicorn

weather_app = FastAPI()
_current_temperature = 20.0  # initial value in degrees C

@weather_app.get('/temperature')
def get_temperature():
    global _current_temperature
    _current_temperature += random.uniform(-3.0, 3.0)
    _current_temperature = max(10.0, min(30.0, _current_temperature))
    return {'temperature': round(_current_temperature, 1), 'unit': 'celsius'}

def _run_weather_server():
    uvicorn.run(weather_app, host='0.0.0.0', port=8001, log_level='warning')

threading.Thread(target=_run_weather_server, daemon=True).start()
time.sleep(1)
print('Weather station running at http://localhost:8001/temperature')

Weather station running at http://localhost:8001/temperature


## Step 3: Configure and start the ACME CSE

A configuration file is written first to skip the interactive onboarding. The `[console] headless = True` setting prevents the CSE from waiting for keyboard input.

In [3]:
import os
import subprocess
import sys
import signal
import time

# kill any previous acmecse process
result = subprocess.run(['pgrep', '-f', 'acmecse'], capture_output=True, text=True)
for pid in result.stdout.strip().splitlines():
    try:
        os.kill(int(pid), signal.SIGTERM)
        print(f'Stopped previous acmecse process (PID {pid})')
    except Exception:
        pass
time.sleep(2)

os.makedirs('/content/cse-runtime', exist_ok=True)

cse_config = """\
[basic.config]
cseType = IN
cseID = id-in
cseName = cse-in
adminID = CAdmin
dataDirectory = /content/cse-runtime
networkInterface = 0.0.0.0
cseHost = 127.0.0.1
httpPort = 8082
logLevel = warning
databaseType = tinydb
consoleTheme = light

[cse.registration]
allowedAEOriginators = C*,S*

[database.tinydb]
writeDelay = 10

[textui]
startWithTUI = False

[console]
headless = True
"""

with open('/content/cse-runtime/acme.ini', 'w') as f:
    f.write(cse_config)
print('CSE configuration written.')

cse_process = subprocess.Popen(
    [sys.executable, '-m', 'acmecse', '-dir', '/content/cse-runtime'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
print(f'ACME CSE process started (PID {cse_process.pid})')

CSE configuration written.
ACME CSE process started (PID 585)


## Step 4: Wait for the CSE to be ready

This cell polls the CSE endpoint until it responds.

In [4]:
import httpx

CSE_BASE_URL = 'http://localhost:8082/cse-in'

for attempt in range(30):
    try:
        r = httpx.get(CSE_BASE_URL, timeout=2.0)
        if r.status_code in (200, 400):
            print(f'CSE is ready (status {r.status_code})')
            break
    except Exception:
        pass
    print(f'Waiting for CSE... (attempt {attempt + 1}/30)')
    time.sleep(2)
else:
    print('CSE did not respond - check logs in /content/cse-runtime/')

CSE is ready (status 400)


## Step 5: Start the oneM2M AE

The Application Entity registers itself and the temperature container in the CSE, then polls the weather station every 30 seconds and writes each reading as a ContentInstance.

In [5]:
import requests
import uuid

AE_NAME       = 'CAdmin'
AE_API_ID     = 'Nmy-weatherstation.example.com'
CNT_NAME      = 'Temperature'
POLL_INTERVAL = 30  # seconds
ONEM2M_RVI    = '4'
WEATHER_URL   = 'http://localhost:8001/temperature'

def _onem2m_headers(content_type: str) -> dict:
    return {
        'Content-Type': content_type,
        'X-M2M-Origin': AE_NAME,
        'X-M2M-RI': str(uuid.uuid4()),
        'X-M2M-RVI': ONEM2M_RVI,
        'Accept': 'application/json',
    }

def _ae_exists() -> bool:
    r = requests.get(f'{CSE_BASE_URL}/{AE_NAME}',
                     headers=_onem2m_headers('application/json'))
    return r.status_code == 200

def _container_exists() -> bool:
    r = requests.get(f'{CSE_BASE_URL}/{AE_NAME}/{CNT_NAME}',
                     headers=_onem2m_headers('application/json'))
    return r.status_code == 200

def _register_ae():
    body = {'m2m:ae': {'rn': AE_NAME, 'api': AE_API_ID, 'rr': True, 'srv': [ONEM2M_RVI]}}
    r = requests.post(CSE_BASE_URL,
                      headers=_onem2m_headers('application/json;ty=2'), json=body)
    print(f'AE registration: {r.status_code}')

def _register_container():
    body = {'m2m:cnt': {'rn': CNT_NAME}}
    r = requests.post(f'{CSE_BASE_URL}/{AE_NAME}',
                      headers=_onem2m_headers('application/json;ty=3'), json=body)
    print(f'Container registration: {r.status_code}')

def _write_cin(temperature: float):
    body = {'m2m:cin': {'con': {'temperature': temperature, 'unit': 'celsius'}}}
    r = requests.post(f'{CSE_BASE_URL}/{AE_NAME}/{CNT_NAME}',
                      headers=_onem2m_headers('application/json;ty=4'), json=body)
    if r.status_code == 201:
        print(f'ContentInstance written: {temperature} C')
    else:
        print(f'ContentInstance failed: {r.status_code} / {r.text}')

def _ae_loop():
    while True:
        try:
            temp = httpx.get(WEATHER_URL, timeout=5.0).json()['temperature']
            _write_cin(temp)
        except Exception as e:
            print(f'AE loop error: {e}')
        time.sleep(POLL_INTERVAL)

if not _ae_exists():
    _register_ae()
else:
    print(f'AE "{AE_NAME}" already registered.')

if not _container_exists():
    _register_container()
else:
    print(f'Container "{CNT_NAME}" already exists.')

threading.Thread(target=_ae_loop, daemon=True).start()
print(f'AE started - writing temperature every {POLL_INTERVAL}s')

AE "CAdmin" already registered.
Container registration: 201
AE started - writing temperature every 30s


## Step 6: Start the MCP Server

The MCP server exposes a single tool `get_temperature` that reads the latest ContentInstance from the CSE and returns the temperature value.

In [6]:
!pip install fastmcp --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 633.8/633.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 23.0 MB/s eta 0:00:00


In [7]:
from mcp.server.fastmcp import FastMCP
import uuid
import requests
import threading

mcp = FastMCP(
    name='Temperature_Retriever',
    host='0.0.0.0',
    port=8050,
)

def _get_latest_cin() -> dict | None:
    """Fetch the latest ContentInstance from the CSE."""
    url = f'{CSE_BASE_URL}/{AE_NAME}/{CNT_NAME}/la'
    headers = {
        'X-M2M-Origin': AE_NAME,
        'X-M2M-RI': str(uuid.uuid4()),
        'X-M2M-RVI': ONEM2M_RVI,
        'Accept': 'application/json',
    }
    r = requests.get(url, headers=headers)
    if r.status_code == 200:
        return r.json()
    print(f'Failed to fetch ContentInstance: {r.status_code}')
    return None

@mcp.tool()
def get_temperature() -> dict:
    """Returns the current temperature from the oneM2M CSE.
    The measurement is in degrees Celsius."""
    cin = _get_latest_cin()
    if cin and 'm2m:cin' in cin:
        return cin['m2m:cin'].get('con', {})
    return {'error': 'No ContentInstance found'}

def _run_mcp_server():
    mcp.run(transport='sse')

threading.Thread(target=_run_mcp_server, daemon=True).start()
time.sleep(1)
print('MCP server running on http://localhost:8050/sse')

INFO:     Started server process [326]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8050 (Press CTRL+C to quit)


MCP server running on http://localhost:8050/sse


## Step 7: Test the MCP tool directly

Call the tool function directly to verify it returns the latest temperature from the CSE.

In [8]:
# verify the MCP tool returns data from the CSE
result = get_temperature()
print(f'MCP tool result: {result}')

MCP tool result: {'temperature': 18.5, 'unit': 'celsius'}


## Step 8: Install ADK

Install the Google Agent Development Kit.

In [9]:
!pip install google-adk --quiet

## Step 9: Configure Gemini API Key

The agent uses the Gemini API. Enter your API key from [Google AI Studio](https://aistudio.google.com/app/apikey).

In [10]:
import os

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('API key loaded from Colab secrets.')
except ImportError:
    import getpass
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter Gemini API key: ')
    print('API key set.')

API key loaded from Colab secrets.


## Step 10: Define and run the ADK Agent

The agent connects to the MCP server via SSE and uses the `get_temperature` tool to answer questions about the current temperature.

In [11]:
import logging
logging.getLogger('uvicorn').setLevel(logging.WARNING)
logging.getLogger('uvicorn.access').setLevel(logging.CRITICAL)
logging.getLogger('httpx').setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset, SseConnectionParams
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import asyncio

MCP_SERVER_URL = 'http://127.0.0.1:8050/sse'

try:
    mcp_toolset = McpToolset(
        connection_params=SseConnectionParams(url=MCP_SERVER_URL)
    )
except Exception as e:
    print(f'Failed to connect to MCP server: {e}')
    mcp_toolset = None

root_agent = LlmAgent(
    model='gemini-2.5-flash',
    name='weather_agent',
    instruction=(
        'You are a helpful assistant that monitors temperature readings '
        'from a simulated weather station via a oneM2M IoT platform. '
        'Always use the get_temperature tool to retrieve the current temperature. '
        'Report the value clearly in degrees Celsius. '
        'If the temperature is below 15 degrees Celsius, advise the user '
        'to close the windows and doors to keep the room warm. '
        'If the temperature is above 20 degrees Celsius, advise the user '
        'to drink plenty of water and stay hydrated. '
        'Always give practical and friendly advice based on the temperature.'
    ),
    tools=[mcp_toolset] if mcp_toolset else [],
)

session_service = InMemorySessionService()
runner = Runner(
    agent=root_agent,
    app_name='weather_demo',
    session_service=session_service,
)

async def ask(question: str):
    session = await session_service.create_session(
        app_name='weather_demo', user_id='user'
    )
    content = types.Content(role='user', parts=[types.Part(text=question)])
    async for event in runner.run_async(
        user_id='user',
        session_id=session.id,
        new_message=content,
    ):
        if event.is_final_response():
            print(event.content.parts[0].text)

await ask('What is the current temperature?')

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.BASE_AUTHENTICATED_TOOL is enabled.
  check_feature_enabled()


The current temperature is 21.2 degrees Celsius. It's quite warm, so remember to drink plenty of water and stay hydrated!
